## Hybrid Retriever


In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma


#utility
import numpy as np
from typing import List,Dict,Any
from sentence_transformers import SentenceTransformer

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.runnables import (RunnablePassthrough,RunnableMap)
from langchain_core.output_parsers import StrOutputParser
import os



C:\Users\kanha\AppData\Local\Temp\ipykernel_27924\3450054094.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
import numpy as np
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

In [ ]:
docs=[
    Document(page_content="Langchain help build LLM Application"),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Effiel Tower is located in Paris"),
    Document(page_content="langchain can used to develop agentic ai"),
    Document(page_content="langchain has many types of retrievers")


]
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"

)
## Dense retruever
vs=FAISS.from_documents(docs,embeddings)
retriever=vs.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
### Sparse retriever(BM25)
sparse_retriever=BM25Retriever.from_documents(docs)
sparse_retriever.k=3

hybrid_retriever=EnsembleRetriever(
    retrievers=[retriever,sparse_retriever],
    weights=[0.7,0.3]
)



In [5]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F2B4F33C50>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001F2EA575750>, k=3)], weights=[0.7, 0.3])

In [6]:
query="How can I build an application using LLMs?"
results=hybrid_retriever.invoke(query)
for i,doc in enumerate(results):
    print(f"\n Document {i+1}.....:\n{doc.page_content}")


 Document 1.....:
Langchain help build LLM Application

 Document 2.....:
langchain can used to develop agentic ai

 Document 3.....:
langchain has many types of retrievers

 Document 4.....:
Pinecone is a vector database for semantic search.


## RAG Pipeline with hybrid retriever

In [ ]:
template="""
Ansert the question based on following context:
{context}
Question:{input}
"""
prompt=PromptTemplate.from_template(template)
llm=init_chat_model(model="groq:llama-3.3-70b-versatile",api_key="")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F2B5F3CD90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F2B5FDFA90>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [13]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [14]:
### Create stiff document chain
document_chain=create_stuff_documents_chain(llm=llm,prompt=prompt)

rag_chain=create_retrieval_chain(hybrid_retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F2B4F33C50>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001F2EA575750>, k=3)], weights=[0.7, 0.3]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnsert the question based on following context:\n{context}\nQuestion:{input}\n')
            | ChatGroq(metadata={'lc_versions':

In [15]:
query={"input":"How can i build an app using LLMs?"}
response=rag_chain.invoke(query)
print(f"Answer:\n {response['answer']}")

for i,doc in enumerate(response['context']):
    print(f"\n Doc {i+1} :{doc.page_content}")

Answer:
 You can build an app using LLMs (Large Language Models) with the help of Langchain. Langchain is a framework that enables you to develop LLM applications, including agentic AI, which can be used to create a wide range of applications. Additionally, Langchain provides various types of retrievers that can be used in conjunction with vector databases like Pinecone for semantic search, making it easier to build and deploy LLM-powered apps.

 Doc 1 :Langchain help build LLM Application

 Doc 2 :langchain can used to develop agentic ai

 Doc 3 :langchain has many types of retrievers

 Doc 4 :Pinecone is a vector database for semantic search.
